# 1. INTRODUCCIÓN

(...)

# 2. EXTRACCIÓN DE DATOS

In [1]:
import os
import requests
import time
from dotenv import load_dotenv
from datetime import datetime, timedelta
import pandas as pd

load_dotenv()
COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY")

CRYPTOS = [
    { "name": "Bitcoin", "binance": "BTCUSDT", "coingecko": "bitcoin" },
    { "name": "Ethereum", "binance": "ETHUSDT", "coingecko": "ethereum" },
    { "name": "Solana", "binance": "SOLUSDT", "coingecko": "solana" },
    { "name": "BinanceCoin", "binance": "BNBUSDT", "coingecko": "binancecoin" },
    { "name": "Ripple", "binance": "XRPUSDT", "coingecko": "ripple" },
]

EXTRACTION_CONFIG = {
    "binance": {
        "url": "https://api.binance.com/api/v3/klines",
        "params": {
            "interval": "1h",
            "startTime": int((datetime.now() - timedelta(days=365)).timestamp() * 1000),
            "endTime": int(datetime.now().timestamp() * 1000)
        }
    },
    "coingecko": {
        "url": "https://api.coingecko.com/api/v3/coins/{id}/market_chart",
        "params": {
            "vs_currency": "usd",
            "days": 365,
            "x_cg_demo_api_key": COINGECKO_API_KEY
        }
    }
}

def extract_binance_full_history(symbol, start_time, end_time, interval="1h"):
    """Paginación para obtener más de 1000 velas de Binance"""
    url = EXTRACTION_CONFIG["binance"]["url"]
    all_data = []
    limit = 1000
    current_start = start_time

    while current_start < end_time:
        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": current_start,
            "endTime": end_time,
            "limit": limit
        }

        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        if not data:
            break

        all_data.extend(data)

        # siguiente batch (última vela + 1ms)
        current_start = data[-1][0] + 1

        time.sleep(0.2)  # evitar rate limit

    return all_data


def extract_data(url, params):
    try:
        response = requests.get(url=url, params=params)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print("Ha ocurrido un error: ", e)


dfs_cryptos = {}

for crypto in CRYPTOS:

    crypto_name = crypto["name"]
    crypto_symbol_binance = crypto["binance"]
    crypto_symbol_coingecko = crypto["coingecko"]

    print(f"---- Procesando {crypto_name}... ----\n")

    # ---------------- BINANCE (PAGINADO) ----------------
    data_binance = extract_binance_full_history(
        crypto_symbol_binance,
        EXTRACTION_CONFIG["binance"]["params"]["startTime"],
        EXTRACTION_CONFIG["binance"]["params"]["endTime"]
    )

    df_binance = pd.DataFrame(data_binance, columns=[
        "open_time", "open_price", "high_price", "low_price",
        "close_price", "volume", "close_time",
        "quote_asset_volume", "number_trades",
        "taker_buy_base_asset_volume",
        "taker_buy_quote_asset_volume",
        "unused_field"
    ])

    df_binance = df_binance[
        ["open_time", "open_price", "high_price", "low_price",
         "close_price", "volume", "close_time"]
    ]

    df_binance["open_time"] = pd.to_datetime(df_binance["open_time"], unit="ms")


    # ---------------- COINGECKO ----------------
    coingecko_url = EXTRACTION_CONFIG["coingecko"]["url"].format(
        id=crypto_symbol_coingecko
    )

    data_coingecko = extract_data(coingecko_url, EXTRACTION_CONFIG["coingecko"]["params"])
    data_coingecko = data_coingecko["market_caps"]

    df_coingecko = pd.DataFrame(data_coingecko, columns=["open_time", "market_cap"])
    df_coingecko["open_time"] = pd.to_datetime(df_coingecko["open_time"], unit="ms")

    df_coingecko = df_coingecko.set_index("open_time").sort_index()
    df_coingecko = df_coingecko.resample("1h").ffill()


    # ---------------- ALINEACIÓN ----------------
    start = max(df_binance["open_time"].min(), df_coingecko.index.min())
    end = min(df_binance["open_time"].max(), df_coingecko.index.max())

    df_binance = df_binance[
        (df_binance["open_time"] >= start) &
        (df_binance["open_time"] <= end)
    ]

    df_coingecko = df_coingecko.loc[start:end]

    df_final = df_binance.merge(
        df_coingecko,
        left_on="open_time",
        right_index=True,
        how="left"
    )

    df_final["crypto"] = crypto_name
    df_final = df_final[["crypto"] + [col for col in df_final.columns if col != "crypto"]] # Ordenar para que el nombre de la crypto sea la primera columna
    dfs_cryptos[crypto_name] = df_final
    print(f"---- Obtención de datos de {crypto_name} ✅ ----\n")


print("---- 🧪 DATASETS FINALES 🧪 ----\n")

for name, df_crypto in dfs_cryptos.items():
    print(f"---- Datos finales obtenidos para el DataFrame de: {name} ----\n")
    display(df_crypto.head(10)) 
    print(df_crypto.shape)

---- Procesando Bitcoin... ----

---- Obtención de datos de Bitcoin ✅ ----

---- Procesando Ethereum... ----

---- Obtención de datos de Ethereum ✅ ----

---- Procesando Solana... ----

---- Obtención de datos de Solana ✅ ----

---- Procesando BinanceCoin... ----

---- Obtención de datos de BinanceCoin ✅ ----

---- Procesando Ripple... ----

---- Obtención de datos de Ripple ✅ ----

---- 🧪 DATASETS FINALES 🧪 ----

---- Datos finales obtenidos para el DataFrame de: Bitcoin ----



,crypto,open_time,open_price,high_price,low_price,close_price,volume,close_time,market_cap
9,Bitcoin,2025-05-03 00:00:00,96887.13000000,96935.67000000,96612.87000000,96700.45000000,559.13625000,1746233999999,1.923252e+12
10,Bitcoin,2025-05-03 01:00:00,96700.46000000,96722.40000000,96435.16000000,96493.94000000,462.40221000,1746237599999,1.923252e+12
11,Bitcoin,2025-05-03 02:00:00,96493.93000000,96699.98000000,96487.07000000,96533.51000000,275.57053000,1746241199999,1.923252e+12
12,Bitcoin,2025-05-03 03:00:00,96533.50000000,96535.96000000,96300.02000000,96337.50000000,761.61563000,1746244799999,1.923252e+12
13,Bitcoin,2025-05-03 04:00:00,96337.50000000,96589.20000000,96307.88000000,96519.05000000,252.68819000,1746248399999,1.923252e+12
14,Bitcoin,2025-05-03 05:00:00,96519.04000000,96593.00000000,96407.97000000,96417.14000000,306.91209000,1746251999999,1.923252e+12
15,Bitcoin,2025-05-03 06:00:00,96417.15000000,96417.15000000,96132.67000000,96183.50000000,539.08747000,1746255599999,1.923252e+12
16,Bitcoin,2025-05-03 07:00:00,96183.50000000,96391.70000000,96157.10000000,96322.68000000,410.18653000,1746259199999,1.923252e+12
17,Bitcoin,2025-05-03 08:00:00,96322.68000000,96421.91000000,96150.00000000,96281.00000000,599.65552000,1746262799999,1.923252e+12
18,Bitcoin,2025-05-03 09:00:00,96281.00000000,96396.23000000,96173.00000000,96254.01000000,318.86883000,1746266399999,1.923252e+12


(8751, 9)
---- Datos finales obtenidos para el DataFrame de: Ethereum ----



,crypto,open_time,open_price,high_price,low_price,close_price,volume,close_time,market_cap
9,Ethereum,2025-05-03 00:00:00,1842.20000000,1843.05000000,1836.20000000,1838.70000000,5360.34490000,1746233999999,2.222690e+11
10,Ethereum,2025-05-03 01:00:00,1838.69000000,1840.50000000,1826.58000000,1829.00000000,10615.55800000,1746237599999,2.222690e+11
11,Ethereum,2025-05-03 02:00:00,1829.00000000,1835.74000000,1828.73000000,1833.89000000,7846.75960000,1746241199999,2.222690e+11
12,Ethereum,2025-05-03 03:00:00,1833.90000000,1837.99000000,1831.20000000,1834.80000000,4783.08330000,1746244799999,2.222690e+11
13,Ethereum,2025-05-03 04:00:00,1834.80000000,1839.30000000,1833.80000000,1838.00000000,5000.67800000,1746248399999,2.222690e+11
14,Ethereum,2025-05-03 05:00:00,1838.00000000,1838.50000000,1833.00000000,1833.19000000,3554.88050000,1746251999999,2.222690e+11
15,Ethereum,2025-05-03 06:00:00,1833.18000000,1833.42000000,1824.60000000,1827.28000000,11042.97150000,1746255599999,2.222690e+11
16,Ethereum,2025-05-03 07:00:00,1827.28000000,1829.89000000,1825.00000000,1826.41000000,6601.68650000,1746259199999,2.222690e+11
17,Ethereum,2025-05-03 08:00:00,1826.41000000,1830.00000000,1822.32000000,1825.30000000,9124.90180000,1746262799999,2.222690e+11
18,Ethereum,2025-05-03 09:00:00,1825.31000000,1828.41000000,1819.26000000,1827.40000000,10881.23930000,1746266399999,2.222690e+11


(8751, 9)
---- Datos finales obtenidos para el DataFrame de: Solana ----



...

(8751, 9)
---- Datos finales obtenidos para el DataFrame de: BinanceCoin ----



,crypto,open_time,open_price,high_price,low_price,close_price,volume,close_time,market_cap
9,BinanceCoin,2025-05-03 00:00:00,600.69000000,601.20000000,599.19000000,599.40000000,4611.80200000,1746233999999,8.772653e+10
10,BinanceCoin,2025-05-03 01:00:00,599.39000000,599.65000000,598.18000000,598.49000000,4418.64600000,1746237599999,8.772653e+10
11,BinanceCoin,2025-05-03 02:00:00,598.48000000,600.09000000,598.41000000,599.73000000,4672.13300000,1746241199999,8.772653e+10
12,BinanceCoin,2025-05-03 03:00:00,599.73000000,600.19000000,598.84000000,599.29000000,2793.24100000,1746244799999,8.772653e+10
13,BinanceCoin,2025-05-03 04:00:00,599.28000000,600.15000000,599.28000000,600.03000000,2227.76800000,1746248399999,8.772653e+10
14,BinanceCoin,2025-05-03 05:00:00,600.03000000,600.11000000,599.62000000,599.89000000,1788.16300000,1746251999999,8.772653e+10
15,BinanceCoin,2025-05-03 06:00:00,599.89000000,599.97000000,598.25000000,598.79000000,5086.40300000,1746255599999,8.772653e+10
16,BinanceCoin,2025-05-03 07:00:00,598.80000000,598.88000000,597.39000000,597.47000000,4517.91200000,1746259199999,8.772653e+10
17,BinanceCoin,2025-05-03 08:00:00,597.48000000,597.66000000,596.25000000,597.07000000,4407.30600000,1746262799999,8.772653e+10
18,BinanceCoin,2025-05-03 09:00:00,597.06000000,597.10000000,596.15000000,596.49000000,4038.50800000,1746266399999,8.772653e+10


(8751, 9)
---- Datos finales obtenidos para el DataFrame de: Ripple ----



,crypto,open_time,open_price,high_price,low_price,close_price,volume,close_time,market_cap
9,Ripple,2025-05-03 00:00:00,2.20930000,2.21320000,2.20110000,2.20610000,1769677.80000000,1746233999999,1.292054e+11
10,Ripple,2025-05-03 01:00:00,2.20620000,2.20790000,2.19700000,2.20240000,1608494.00000000,1746237599999,1.292054e+11
11,Ripple,2025-05-03 02:00:00,2.20240000,2.22000000,2.20210000,2.21860000,2413644.80000000,1746241199999,1.292054e+11
12,Ripple,2025-05-03 03:00:00,2.21850000,2.21980000,2.21100000,2.21230000,1920666.00000000,1746244799999,1.292054e+11
13,Ripple,2025-05-03 04:00:00,2.21230000,2.21870000,2.21060000,2.21730000,1112240.50000000,1746248399999,1.292054e+11
14,Ripple,2025-05-03 05:00:00,2.21730000,2.21740000,2.21010000,2.21180000,744724.70000000,1746251999999,1.292054e+11
15,Ripple,2025-05-03 06:00:00,2.21190000,2.21220000,2.20340000,2.20440000,985756.60000000,1746255599999,1.292054e+11
16,Ripple,2025-05-03 07:00:00,2.20440000,2.20510000,2.20000000,2.20340000,1436514.80000000,1746259199999,1.292054e+11
17,Ripple,2025-05-03 08:00:00,2.20350000,2.20740000,2.19810000,2.20390000,1854425.30000000,1746262799999,1.292054e+11
18,Ripple,2025-05-03 09:00:00,2.20390000,2.20410000,2.19410000,2.19950000,1417229.90000000,1746266399999,1.292054e+11


(8751, 9)


# 3. ANÁLISIS EXPLORATORIO Y DIAGNÓSTICO DE LA CALIDAD DE LOS DATOS

In [2]:
# Para cada DataFrame, realizar un diagnóstico para identificar problemas de calidad, como tipos de datos incorrectos, valores nulos, etc.

def quality_check(df, df_name):
    """ Función para dignosticar la estructura y la calidad de los datos provenientes de cada DataFrame """
    
    print(f"\n{'='*60}")
    print(f"DIAGNÓSTICO DEL DATAFRAME: {df_name}")
    print(f"{'='*60}")
    
    # Información general
    print("\n--- INFO GENERAL ---")
    df.info()
    
    # Conteo de nulos reales
    print("\n--- NULOS REALES (isnull) ---")
    print(df.isnull().sum())
    
    print("\n--- % DE NULOS ---")
    print((df.isnull().mean() * 100).round(2))
    
    # Valores sospechosos típicos
    print("\n--- VALORES SOSPECHOSOS ---")
    valores_sospechosos = ["-", "null", "NULL", "None", "", " "]
    
    for val in valores_sospechosos:
        conteo = (df == val).sum().sum()
        if conteo > 0:
            print(f"Valor '{val}' encontrado {conteo} veces")
    
    # Detección de tipos incorrectos
    print("\n--- COLUMNAS OBJECT (posibles tipos incorrectos) ---")
    print(df.select_dtypes(include=["object", "string"]).columns.tolist())

for name, df_crypto in dfs_cryptos.items():
    quality_check(df_crypto, name)


DIAGNÓSTICO DEL DATAFRAME: Bitcoin

--- INFO GENERAL ---
<class 'pandas.DataFrame'>
RangeIndex: 8751 entries, 9 to 8759
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   crypto       8751 non-null   str           
 1   open_time    8751 non-null   datetime64[ms]
 2   open_price   8751 non-null   str           
 3   high_price   8751 non-null   str           
 4   low_price    8751 non-null   str           
 5   close_price  8751 non-null   str           
 6   volume       8751 non-null   str           
 7   close_time   8751 non-null   int64         
 8   market_cap   8751 non-null   float64       
dtypes: datetime64[ms](1), float64(1), int64(1), str(6)
memory usage: 1.3 MB

--- NULOS REALES (isnull) ---
crypto         0
open_time      0
open_price     0
high_price     0
low_price      0
close_price    0
volume         0
close_time     0
market_cap     0
dtype: int64

--- % DE NULOS ---
crypto        

Con los resultados obtenidos, se observa que no existen valores nulos en el dataset, lo que indica que los datos extraídos de las APIs presentan un buen nivel de consistencia.

En cuanto a la conversión de tipos de datos, es necesario realizar los siguientes ajustes:

* Las variables `open_price`, `high_price`, `low_price`, `close_price` y `volume` deben convertirse a tipo float64, con el objetivo de preservar los valores decimales y mantener la máxima precisión en los cálculos posteriores.
* La variable `close_time` debe convertirse a tipo datetime, lo que permitirá un manejo adecuado de las fechas y facilitará su uso en análisis temporales.

# 4. TRANSFORMACIÓN Y LIMPIEZA DE LOS DATOS

In [3]:
# Transform:
## Conversión de tipos

numeric_cols = ["open_price", "high_price", "low_price", "close_price", "volume"]
dfs_correct_types = {}

def convert_dtypes(df, df_name):
    """ Función para convertir los tipos correctamente en cada DataFrame """
    
    print(f"\n{'='*60}")
    print(f"CONVERSIÓN DE TIPOS DEL DATAFRAME: {df_name}")
    print(f"{'='*60}")

    if "close_time" in df.columns:
        df["close_time"] = pd.to_datetime(df["close_time"], errors="coerce", unit="ms")

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

    # Revisamos cómo está ahora cada DataFrame
    print("\n--- INFO GENERAL POSTERIOR ---")
    df.info()

    dfs_correct_types[df_name] = df

for name, df_crypto in dfs_cryptos.items():
    convert_dtypes(df_crypto, name)

print("\nFINALIZADA CON ÉXITO LA CONVERSIÓN DE TIPOS ✅")
print(f"\n🧪 DataFrames con tipos correctos disponibles en 'dfs_correct_types': {list(dfs_correct_types.keys())}")


CONVERSIÓN DE TIPOS DEL DATAFRAME: Bitcoin

--- INFO GENERAL POSTERIOR ---
<class 'pandas.DataFrame'>
RangeIndex: 8751 entries, 9 to 8759
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   crypto       8751 non-null   str           
 1   open_time    8751 non-null   datetime64[ms]
 2   open_price   8751 non-null   float64       
 3   high_price   8751 non-null   float64       
 4   low_price    8751 non-null   float64       
 5   close_price  8751 non-null   float64       
 6   volume       8751 non-null   float64       
 7   close_time   8751 non-null   datetime64[ms]
 8   market_cap   8751 non-null   float64       
dtypes: datetime64[ms](2), float64(6), str(1)
memory usage: 676.3 KB

CONVERSIÓN DE TIPOS DEL DATAFRAME: Ethereum

--- INFO GENERAL POSTERIOR ---
<class 'pandas.DataFrame'>
RangeIndex: 8751 entries, 9 to 8759
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         


# 5. ALMACENAMIENTO DE LOS DATOS PARA EL DATA LAKE

In [4]:
# Load:
## Guardar todos los DataFrames obtenidos, incluido uno final con TODOS los datos de las CRYPTOS

output_dir = "../data/processed/data_lake"
os.makedirs(output_dir, exist_ok=True)

# DataFrame final que contiene los datos de TODAS las cryptos
df_final = pd.concat(dfs_correct_types.values(), ignore_index=True)
print("\n--- INFO GENERAL DEL DATAFRAME CON TODAS LAS COINS ---")
df_final.info()

def load_data(df, df_name):
    """ Función para almacenar los datos en un Data Lake """

    print(f"\n{'='*60}")
    print(f"ALMACENANDO EL DATAFRAME DE: {df_name}")
    print(f"{'='*60}")

    df.to_parquet(
        f"{output_dir}/{df_name.lower()}.parquet",
        index=False,
        engine="pyarrow"
    )

    print(f"\nDatos de {df_name} almacenados correctamente ✅")


for name, df_crypto in dfs_correct_types.items():
    load_data(df_crypto, name)

load_data(df_final, "all_cryptos")
print("\nFINALIZADA CON ÉXITO LA CREACIÓN DEL DATA LAKE ✅")


--- INFO GENERAL DEL DATAFRAME CON TODAS LAS COINS ---
<class 'pandas.DataFrame'>
RangeIndex: 43755 entries, 0 to 43754
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   crypto       43755 non-null  str           
 1   open_time    43755 non-null  datetime64[ms]
 2   open_price   43755 non-null  float64       
 3   high_price   43755 non-null  float64       
 4   low_price    43755 non-null  float64       
 5   close_price  43755 non-null  float64       
 6   volume       43755 non-null  float64       
 7   close_time   43755 non-null  datetime64[ms]
 8   market_cap   43755 non-null  float64       
dtypes: datetime64[ms](2), float64(6), str(1)
memory usage: 3.3 MB

ALMACENANDO EL DATAFRAME DE: Bitcoin

Datos de Bitcoin almacenados correctamente ✅

ALMACENANDO EL DATAFRAME DE: Ethereum

Datos de Ethereum almacenados correctamente ✅

ALMACENANDO EL DATAFRAME DE: Solana

Datos de Solana almacenados corre